In [21]:
#--- IMPORTS ---

import pandas as pd
import numpy as np
import glob
import os
from scipy.signal import find_peaks

In [30]:
# Tuning

USE_DUMMY_DATA = True
EXPORT_TO_CSV = True

SEGMENT_FOLDER_NAME = "Sections"
SEGMENT_FILENAME_PREFIX = "sections_"

DUMMY_LAPS_PER_TRACK = 3

DUMMY_MAX_SPEED_KPH = 280
DUMMY_ACCEL_RATE = 0.5
DUMMY_BRAKE_RATE = 1.6
DUMMY_COAST_RATE = 0.05
DUMMY_LOW_SPEED_OVERRIDE_KPH = 50

DUMMY_BRAKE_PROBABILITY = 0.8   # percentage of steering & braking corners
DUMMY_MAX_BRAKE_PRESSURE = 55
DUMMY_BRAKE_PEAK_POS = 0.10

DUMMY_STEERING_NOISE = 1 # Amount of random 'wobble' in steering. Higher is messier.
DUMMY_BRAKE_NOISE = 0.8   # Amount of random fluctuation in brake pressure.
MINIMUM_EVENT_DURATION_M = 10 # meters
MINIMUM_DISTANCE_BETWEEN_CORNERS = 30 # meters

DUMMY_DAMPER_LONG_G_SENSITIVITY = 15
DUMMY_DAMPER_LAT_G_SENSITIVITY = 20
DUMMY_AERO_SENSITIVITY = 0.01

BRAKE_COLUMN_NAME = 'pBrake Front'
STEERING_COLUMN_NAME = 'aSteering'

# values above which it is considered an event
BRAKE_THRESHOLD = 5
STEERING_THRESHOLD = 15

In [31]:
def generate_dummy_telemetry(base_dir, num_laps, max_speed, accel, brake_rate, coast_rate, low_speed_override,
                             brake_prob, max_brake_pressure, brake_peak_pos, long_g_sens, lat_g_sens, steer_noise,
                             brake_noise, aero_sensitivity):
    """Generates dynamically plausible telemetry by simulating a vehicle lap."""
    print("Starting physics-based dummy data generation...")
    coord_path = os.path.join(base_dir, 'Coordinates')
    telemetry_base_path = os.path.join(base_dir, 'Telemetry', 'Dummy')
    coordinate_files = glob.glob(f'{coord_path}/*.csv')
    if not coordinate_files:
        print(f"Warning: No coordinate files found in '{coord_path}'. Cannot generate dummy data.")
        return
    for coord_file in coordinate_files:
        track_name = os.path.basename(coord_file).replace('.csv', '')
        print(f'  -> Simulating laps for track: {track_name}')
        track_telemetry_folder = os.path.join(telemetry_base_path, track_name)
        os.makedirs(track_telemetry_folder, exist_ok=True)
        coords_df = pd.read_csv(coord_file)
        if 'Distance' not in coords_df.columns:
            distances = np.sqrt(np.diff(coords_df['x'])**2 + np.diff(coords_df['y'])**2)
            coords_df['Distance'] = np.insert(np.cumsum(distances), 0, 0)
        headings = np.arctan2(np.diff(coords_df['y']), np.diff(coords_df['x']))
        curvatures = np.abs(np.diff(np.unwrap(headings)))
        curvatures = np.insert(curvatures, 0, 0)
        curvatures = np.append(curvatures, 0)
        corner_indices, _ = find_peaks(curvatures, height=0.015, distance=150)
        for lap_num in range(1, num_laps + 1):
            num_points = len(coords_df)
            lap = pd.DataFrame({'Distance': coords_df['Distance'], 'Time': np.zeros(num_points), 'Car Speed': np.zeros(num_points), 'pBrake Front': np.zeros(num_points), 'rThrottle': np.zeros(num_points), 'aSteering': np.zeros(num_points), 'xDamperFL': np.zeros(num_points), 'xDamperFR': np.zeros(num_points), 'xDamperRL': np.zeros(num_points), 'xDamperRR': np.zeros(num_points)})
            for i in range(1, num_points):
                dist_step = lap.at[i, 'Distance'] - lap.at[i-1, 'Distance']
                if dist_step <= 0:
                    lap.loc[i] = lap.loc[i-1]; lap.at[i, 'Distance'] = lap.at[i-1, 'Distance'] + 0.1; dist_step = 0.1
                prev_speed = lap.at[i-1, 'Car Speed']
                is_braking_zone = any(0 < (c - i) < 80 for c in corner_indices)
                throttle_intent, brake_intent = 100.0, 0.0
                if is_braking_zone and np.random.rand() < brake_prob:
                    throttle_intent = 0.0
                    peak = np.random.uniform(max_brake_pressure * 0.8, max_brake_pressure)
                    dist_to_corner = min(c - i for c in corner_indices if c > i)
                    brake_intent = np.interp(dist_to_corner, [0, 80 * (1-brake_peak_pos), 80], [0, peak, 0])
                if prev_speed < low_speed_override: throttle_intent, brake_intent = 100.0, 0.0
                # Add noise to the brake intent, but only if braking is actually happening
                if brake_intent > 0:
                    brake_intent += np.random.normal(0, brake_noise)
                    brake_intent = max(0, brake_intent) # Ensure noise doesn't make it negative

                # The base steering is calculated from curvature
                steer_intent = curvatures[i] * 1000 * (1 if np.random.rand() > 0.5 else -1)
                # Add the tunable random "wobble" noise
                steer_intent += np.random.normal(0, steer_noise)

                # --- Assign final values to the DataFrame ---
                #lap.at[i, 'rThrottle'] = throttle_intent
                #lap.at[i, 'pBrake Front'] = brake_intent
                #lap.at[i, 'aSteering'] = steer_intent
                lap.at[i, 'rThrottle'], lap.at[i, 'pBrake Front'], lap.at[i, 'aSteering'] = throttle_intent, brake_intent, curvatures[i] * 1000 * (1 + np.random.normal(0, 0.1)) * (1 if np.random.rand() > 0.5 else -1)
                speed_gain = throttle_intent / 100 * accel * (1 - prev_speed / max_speed)
                speed_loss_brake, speed_loss_coast = brake_intent / 100 * brake_rate, coast_rate
                current_speed = max(0, prev_speed + speed_gain - speed_loss_brake - speed_loss_coast)
                lap.at[i, 'Car Speed'] = current_speed
                lap.at[i, 'Time'] = lap.at[i-1, 'Time'] + (dist_step / (current_speed / 3.6 + 1e-6))
                g_lon, g_lat = (current_speed - prev_speed) * long_g_sens, (current_speed**2 / (1000 + 1e-6)) * lap.at[i, 'aSteering'] * lat_g_sens

                aero_compression = - (current_speed**2) * aero_sensitivity

                lap.at[i, 'xDamperFL'], lap.at[i, 'xDamperFR'], lap.at[i, 'xDamperRL'], lap.at[i, 'xDamperRR'] = -g_lon + g_lat + aero_compression, -g_lon - g_lat + aero_compression, g_lon + g_lat + aero_compression, g_lon - g_lat + aero_compression
            dummy_filename = os.path.join(track_telemetry_folder, f"dummy_lap_{lap_num}.csv")
            lap.to_csv(dummy_filename, index=False)
        print(f'    {num_laps} simulated telemetry files created for {track_name}.')

def get_event_intervals(lap_df, column_name, threshold):
    if 'Steering' in column_name: is_event = lap_df[column_name].abs() > threshold
    else: is_event = lap_df[column_name] > threshold
    is_event_shifted = is_event.shift(1).fillna(False)
    starts = lap_df[is_event & ~is_event_shifted]['Distance']
    ends = lap_df[~is_event & is_event_shifted]['Distance']
    if len(starts) > len(ends):
        ends = pd.concat([ends, pd.Series([lap_df['Distance'].iloc[-1]])], ignore_index=True)
    return list(zip(starts, ends))

def merge_overlapping_intervals(intervals):
    if not intervals: return []
    intervals.sort(key=lambda x: x[0])
    merged = [intervals[0]]
    for current_start, current_end in intervals[1:]:
        _, last_merged_end = merged[-1]
        if current_start < last_merged_end:
            merged[-1] = (merged[-1][0], max(last_merged_end, current_end))
        else:
            merged.append((current_start, current_end))
    return merged

def merge_proximal_intervals(intervals, min_gap):
    """
    Merges intervals that are separated by a gap smaller than min_gap.
    """
    if not intervals:
        return []
    
    # It's assumed intervals are already sorted by start time
    merged = [intervals[0]]
    for current_start, current_end in intervals[1:]:
        last_merged_start, last_merged_end = merged[-1]
        
        # Calculate the gap between the last merged event and the current one
        gap = current_start - last_merged_end
        
        if gap < min_gap:
            # If the gap is small enough, merge them by extending the end
            merged[-1] = (last_merged_start, max(last_merged_end, current_end))
        else:
            # If the gap is large, it's a new distinct event
            merged.append((current_start, current_end))
            
    return merged


In [32]:
# Preparation

script_dir = os.getcwd()

if USE_DUMMY_DATA:
    print("--- DUMMY DATA MODE ENABLED ---")
    generate_dummy_telemetry(script_dir, DUMMY_LAPS_PER_TRACK, DUMMY_MAX_SPEED_KPH, DUMMY_ACCEL_RATE, 
                             DUMMY_BRAKE_RATE, DUMMY_COAST_RATE, DUMMY_LOW_SPEED_OVERRIDE_KPH, DUMMY_BRAKE_PROBABILITY, 
                             DUMMY_MAX_BRAKE_PRESSURE, DUMMY_BRAKE_PEAK_POS, DUMMY_DAMPER_LONG_G_SENSITIVITY, 
                             DUMMY_DAMPER_LAT_G_SENSITIVITY, DUMMY_STEERING_NOISE, DUMMY_BRAKE_NOISE, DUMMY_AERO_SENSITIVITY)
    print("--- DUMMY DATA GENERATION COMPLETE ---\n")

# --- Setup Paths Using Tunable Variables ---
if USE_DUMMY_DATA:
    base_telemetry_path = os.path.join(script_dir, 'Telemetry', 'Dummy')
    output_folder = os.path.join(script_dir, SEGMENT_FOLDER_NAME, 'Dummy')
else:
    base_telemetry_path = os.path.join(script_dir, 'Telemetry', 'Real')
    output_folder = os.path.join(script_dir, SEGMENT_FOLDER_NAME, 'Real')
if EXPORT_TO_CSV:
    os.makedirs(output_folder, exist_ok=True)
    print(f"Export folder is '{output_folder}'")

# --- Find Track Folders ---
try:
    track_folders = [f.path for f in os.scandir(base_telemetry_path) if f.is_dir()]
    if not track_folders: raise FileNotFoundError(f"No track folders found at path: {base_telemetry_path}")
    print(f"Found {len(track_folders)} track folders to process.")
except FileNotFoundError as e:
    print(e)
    track_folders = []

overwrite_all = None

--- DUMMY DATA MODE ENABLED ---
Starting physics-based dummy data generation...
  -> Simulating laps for track: silverstone_gp
    3 simulated telemetry files created for silverstone_gp.
  -> Simulating laps for track: donington_gp
    3 simulated telemetry files created for donington_gp.
  -> Simulating laps for track: brands_gp
    3 simulated telemetry files created for brands_gp.
  -> Simulating laps for track: hungaroring
    3 simulated telemetry files created for hungaroring.
  -> Simulating laps for track: monza
    3 simulated telemetry files created for monza.
  -> Simulating laps for track: spa
    3 simulated telemetry files created for spa.
  -> Simulating laps for track: zandvoort
    3 simulated telemetry files created for zandvoort.
--- DUMMY DATA GENERATION COMPLETE ---

Export folder is 'h:\My Drive\Cranfield\11. Individual Thesis\Code\Sections\Dummy'
Found 7 track folders to process.


In [33]:
# --- EXECUTION ---

# --- A SINGLE, UNIFIED MAIN LOOP ---
for track_path in track_folders:
    track_name = os.path.basename(track_path)
    print(f'\n===== PROCESSING TRACK: {track_name.upper()} =====\n')
    
    try:
        file_list = glob.glob(f'{track_path}/**/*.csv', recursive=True)
        if not file_list: raise FileNotFoundError(f"Warning: No CSV files found for track '{track_name}'. Skipping.")
        lap_data = [pd.read_csv(file) for file in file_list]
        max_dist = max(lap['Distance'].max() for lap in lap_data)
        print(f"Loaded {len(lap_data)} lap(s) for {track_name}.")
    except Exception as e:
        print(f"An error occurred loading data for {track_name}: {e}. Skipping.")
        continue

    all_lap_events = []
    for lap_df in lap_data:
        brake_intervals = get_event_intervals(lap_df, BRAKE_COLUMN_NAME, BRAKE_THRESHOLD)
        steer_intervals = get_event_intervals(lap_df, STEERING_COLUMN_NAME, STEERING_THRESHOLD)
        combined_lap_events = brake_intervals + steer_intervals
        merged_lap_events = merge_overlapping_intervals(combined_lap_events)
        all_lap_events.extend(merged_lap_events)
    
    final_corner_intervals = merge_overlapping_intervals(all_lap_events)
    print(f"Found {len(final_corner_intervals)} raw corner zones before proximity merging.")

    final_corner_intervals = merge_proximal_intervals(final_corner_intervals, MINIMUM_DISTANCE_BETWEEN_CORNERS)
    print(f"Found {len(final_corner_intervals)} final corner zones after proximity merging.")

    # --- Step 3: Build Final Segments (ROBUST LOGIC) ---
    definitive_corner_starts = sorted([start for start, end in final_corner_intervals])
    final_segments = []
    start_dist = 0    
    for i, end_dist in enumerate(definitive_corner_starts):
        if end_dist > start_dist:
            segment = {'segment_id': i, 'start_m': start_dist, 'end_m': end_dist}
            final_segments.append(segment)
            start_dist = end_dist
    if start_dist < max_dist:
        final_segments.append({'segment_id': len(definitive_corner_starts), 'start_m': start_dist, 'end_m': max_dist})
    
    segments_df = pd.DataFrame(final_segments)
    
    if len(segments_df) != len(final_corner_intervals) + 1 and len(segments_df) != len(final_corner_intervals):
        print(f"   -> Warning: Mismatch between corners found ({len(final_corner_intervals)}) and segments created ({len(segments_df)}).")


    # --- Step 4: Export to CSV ---
    if EXPORT_TO_CSV:
        print(f"\n--- Exporting section file for {track_name.upper()} ---")
        output_filename = f'{SEGMENT_FILENAME_PREFIX}{track_name}.csv'
        output_path = os.path.join(output_folder, output_filename)
        
        if os.path.exists(output_path):
            if overwrite_all is None:
                choice = input("One or more segment files already exist. Overwrite ALL existing files? (y/n): ")
                overwrite_all = (choice.lower() == 'y')
            if overwrite_all:
                segments_df.to_csv(output_path, index=False)
                print(f'File overwritten: {output_path}')
            else:
                print(f'Export skipped for existing file: {output_path}')
        else:
            segments_df.to_csv(output_path, index=False)
            print(f'File exported: {output_path}')

print("\n===== All tracks processed. Script finished. =====")


===== PROCESSING TRACK: DONINGTON_GP =====

Loaded 3 lap(s) for donington_gp.
Found 37 raw corner zones before proximity merging.
Found 7 final corner zones after proximity merging.

--- Exporting section file for DONINGTON_GP ---
File overwritten: h:\My Drive\Cranfield\11. Individual Thesis\Code\Sections\Dummy\sections_donington_gp.csv

===== PROCESSING TRACK: BRANDS_GP =====

Loaded 3 lap(s) for brands_gp.
Found 88 raw corner zones before proximity merging.
Found 10 final corner zones after proximity merging.

--- Exporting section file for BRANDS_GP ---
File overwritten: h:\My Drive\Cranfield\11. Individual Thesis\Code\Sections\Dummy\sections_brands_gp.csv

===== PROCESSING TRACK: HUNGARORING =====

Loaded 3 lap(s) for hungaroring.
Found 74 raw corner zones before proximity merging.
Found 10 final corner zones after proximity merging.

--- Exporting section file for HUNGARORING ---
File overwritten: h:\My Drive\Cranfield\11. Individual Thesis\Code\Sections\Dummy\sections_hungarorin